In [9]:
import sys
import json
import time
import gc
from pathlib import Path
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from jiwer import cer
from tqdm.auto import tqdm
import pandas as pd
 
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
 
from src.data import (
    parse_transcript_file,
    fix_audio_extension,
    ASRDataset,
    build_vocab_from_df,
    make_collate_fn,
)
from src.model_baseline import HuBERT_CTC, HuBERT_Linear_CTC
from src.train_utils import set_seed, count_parameters, save_checkpoint
from src.eval_utils import run_inference

In [10]:
SSL_MODELS = {
    "mHuBERT-147":   "utter-project/mHuBERT-147-base-3rd-iter",
    "HuBERT-base":   "facebook/hubert-base-ls960",
    "wav2vec2-base":  "facebook/wav2vec2-base",
}

DOWNSTREAM_CONFIGS = {
    "linear":       {"class": "linear"},
    "transformer_2L": {"class": "transformer", "num_layers": 2},
    "transformer_6L": {"class": "transformer", "num_layers": 6},
}

TRAIN_CONFIG = {
    "data_size": "10min",
    "lang": "swa",
    "batch_size": 8,
    "grad_accumulation": 4,
    "learning_rate": 1e-4,
    "weight_decay": 1e-6,
    "target_iterations": 1500,
    "eval_every": 5,
}

DATA_DIR = PROJECT_ROOT / "data" / "commonvoice" / "swa"
RESULTS_DIR = PROJECT_ROOT / "results" / "downstream_scaling"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "downstream_scaling"
 
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [11]:
set_seed(42)
 
df_train = parse_transcript_file(DATA_DIR / "transcript_10min_train.txt")
df_dev = parse_transcript_file(DATA_DIR / "transcript_10min_dev.txt")
df_test = parse_transcript_file(DATA_DIR / "transcript_10min_test.txt")
 
for df in [df_train, df_dev, df_test]:
    fix_audio_extension(df)
 
vocab = build_vocab_from_df(df_train)
idx_to_char = {v: k for k, v in vocab.items()}
collate = make_collate_fn(vocab)
 
AUDIO_DIR = DATA_DIR / "wav"
train_dataset = ASRDataset(df_train, AUDIO_DIR)
dev_dataset = ASRDataset(df_dev, AUDIO_DIR)
test_dataset = ASRDataset(df_test, AUDIO_DIR)
 
train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG["batch_size"],
                          shuffle=True, collate_fn=collate, num_workers=0, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=TRAIN_CONFIG["batch_size"],
                        shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG["batch_size"],
                         shuffle=False, collate_fn=collate, num_workers=0)
 
# Compute number of epochs from target iterations
effective_batch = TRAIN_CONFIG["batch_size"] * TRAIN_CONFIG["grad_accumulation"]
iterations_per_epoch = len(train_dataset) / effective_batch
num_epochs = int((TRAIN_CONFIG["target_iterations"] / iterations_per_epoch) + 0.9999)
print(f"Vocab size: {len(vocab)} | Train: {len(train_dataset)} | Epochs: {num_epochs}")

Parsed 92 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_train.txt
Parsed 103 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_dev.txt
Parsed 100 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_test.txt
Loaded 92 samples from            utt_id     audio_filename  \
0   cv_swa_000000  cv_swa_000000.wav   
1   cv_swa_000001  cv_swa_000001.wav   
2   cv_swa_000002  cv_swa_000002.wav   
3   cv_swa_000003  cv_swa_000003.wav   
4   cv_swa_000004  cv_swa_000004.wav   
..            ...                ...   
87  cv_swa_000087  cv_swa_000087.wav   
88  cv_swa_000088  cv_swa_000088.wav   
89  cv_swa_000089  cv_swa_000089.wav   
90  cv_swa_000090  cv_swa_000090.wav   
91  cv_swa_000091  cv_swa_000091.wav   

                                                 text  
0   wachambuzi 

In [12]:
def train_epoch_silent(model, loader, criterion, optimizer, device, grad_accum=1):
    """Train one epoch — no per-batch progress bar."""
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(loader):
        audios = batch['audios'].to(device)
        texts = batch['texts'].to(device)
        audio_lengths = batch['audio_lengths']
        text_lengths = batch['text_lengths']
        
        logits, T = model(audios, audio_lengths)
        log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
        input_lengths = torch.full((log_probs.size(1),), T, dtype=torch.long)
        
        loss = criterion(log_probs, texts, input_lengths, text_lengths)
        loss = loss / grad_accum
        loss.backward()
        
        if (batch_idx + 1) % grad_accum == 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * grad_accum
    
    if len(loader) % grad_accum != 0:
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
    
    return total_loss / len(loader)

In [13]:
def evaluate_silent(model, loader, criterion, device):
    """Evaluate — no per-batch progress bar."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in loader:
            audios = batch['audios'].to(device)
            texts = batch['texts'].to(device)
            audio_lengths = batch['audio_lengths']
            text_lengths = batch['text_lengths']
            
            logits, T = model(audios, audio_lengths)
            log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
            input_lengths = torch.full((log_probs.size(1),), T, dtype=torch.long)
            
            loss = criterion(log_probs, texts, input_lengths, text_lengths)
            total_loss += loss.item()
    
    return total_loss / len(loader)

In [14]:
def build_model(ssl_name, ssl_hf_name, downstream_name, downstream_cfg, vocab_size):
    """Instantiate the right model class."""
    if downstream_cfg["class"] == "linear":
        model = HuBERT_Linear_CTC(vocab_size=vocab_size, model_name=ssl_hf_name)
    else:
        model = HuBERT_CTC(
            vocab_size=vocab_size,
            model_name=ssl_hf_name,
            num_transformer_layers=downstream_cfg["num_layers"],
        )
    return model

In [15]:
def run_experiment(ssl_name, ssl_hf_name, downstream_name, downstream_cfg):
    """
    Full training + evaluation pipeline for one (SSL, downstream) pair.
    Returns a results dict.
    """
    exp_name = f"{ssl_name}_{downstream_name}"
    print(f"\n{'='*60}")
    print(f"  Experiment: {exp_name}")
    print(f"  SSL: {ssl_name} ({ssl_hf_name})")
    print(f"  Downstream: {downstream_name}")
    print(f"{'='*60}")
    
    set_seed(42)
    start_time = time.time()
    
    # Build model
    model = build_model(ssl_name, ssl_hf_name, downstream_name, downstream_cfg, len(vocab))
    model = model.to(device)
    total_params, trainable_params = count_parameters(model)
    
    # Optimizer
    criterion = nn.CTCLoss(blank=vocab["<blank>"], zero_infinity=True)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10,
    )
    
    # Training loop
    best_dev_loss = float("inf")
    best_ckpt_path = CHECKPOINTS_DIR / f"{exp_name}_best.pt"
    history = {"train_loss": [], "dev_loss": [], "epoch": []}
    
    pbar = tqdm(range(num_epochs), desc=exp_name, unit="epoch")
    for epoch in pbar:
        train_loss = train_epoch_silent(
            model, train_loader, criterion, optimizer,
            device, TRAIN_CONFIG["grad_accumulation"],
        )
        
        # Evaluate periodically
        if (epoch + 1) % TRAIN_CONFIG["eval_every"] == 0 or epoch == 0 or epoch == num_epochs - 1:
            dev_loss = evaluate_silent(model, dev_loader, criterion, device)
            scheduler.step(dev_loss)
        else:
            dev_loss = history["dev_loss"][-1] if history["dev_loss"] else float("inf")
        
        history["train_loss"].append(train_loss)
        history["dev_loss"].append(dev_loss)
        history["epoch"].append(epoch + 1)
        
        pbar.set_postfix(train=f"{train_loss:.3f}", dev=f"{dev_loss:.3f}")
        
        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            save_checkpoint(
                model=model, optimizer=optimizer, epoch=epoch,
                path=best_ckpt_path, dev_loss=dev_loss,
                config={**TRAIN_CONFIG, "ssl": ssl_name, "downstream": downstream_name},
                vocab=vocab,
            )
    
    # Test inference
    checkpoint = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    
    predictions, references = run_inference(
        model=model, loader=test_loader, device=device,
        idx_to_char=idx_to_char, vocab=vocab,
    )
    test_cer = cer(references, predictions)
    elapsed = (time.time() - start_time) / 60
    
    # Save history
    with open(RESULTS_DIR / f"{exp_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    
    # Cleanup GPU memory
    del model, optimizer, scheduler, criterion
    torch.cuda.empty_cache()
    gc.collect()
    
    results = {
        "experiment": exp_name,
        "ssl_model": ssl_name,
        "downstream": downstream_name,
        "total_params_M": round(total_params / 1e6, 1),
        "trainable_params_M": round(trainable_params / 1e6, 2),
        "best_dev_loss": round(best_dev_loss, 4),
        "test_cer": round(test_cer, 4),
        "training_minutes": round(elapsed, 1),
    }
    
    # Save individual result
    with open(RESULTS_DIR / f"{exp_name}_results.json", "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"  ✓ CER = {test_cer:.4f} | Dev loss = {best_dev_loss:.4f} | {elapsed:.1f} min")
    return results

In [16]:
all_results = []
 
# Reload existing results if they already exist
for json_path in RESULTS_DIR.glob("*_results.json"):
    with open(json_path, "r") as f:
        all_results.append(json.load(f))

# Keep track of experiments already done
done_experiments = {r["experiment"] for r in all_results}

print(f"Loaded {len(done_experiments)} existing results.")

# Run only missing experiments
for ssl_name, ssl_hf_name in SSL_MODELS.items():
    for ds_name, ds_cfg in DOWNSTREAM_CONFIGS.items():
        exp_name = f"{ssl_name}_{ds_name}"
        
        if exp_name in done_experiments:
            print(f"Skipping already done: {exp_name}")
            continue
        
        result = run_experiment(ssl_name, ssl_hf_name, ds_name, ds_cfg)
        all_results.append(result)
        done_experiments.add(exp_name)

Loaded 9 existing results.
Skipping already done: mHuBERT-147_linear
Skipping already done: mHuBERT-147_transformer_2L
Skipping already done: mHuBERT-147_transformer_6L
Skipping already done: HuBERT-base_linear
Skipping already done: HuBERT-base_transformer_2L
Skipping already done: HuBERT-base_transformer_6L
Skipping already done: wav2vec2-base_linear
Skipping already done: wav2vec2-base_transformer_2L
Skipping already done: wav2vec2-base_transformer_6L


In [17]:
df_results = pd.DataFrame(all_results)
 
# Save full summary
df_results.to_csv(RESULTS_DIR / "summary_downstream_scaling.csv", index=False)
 
# Display as pivot table: SSL x Downstream → CER
pivot = df_results.pivot(index="ssl_model", columns="downstream", values="test_cer")
pivot = pivot[["linear", "transformer_2L", "transformer_6L"]]  # order columns
print("\n" + "="*60)
print("  CER by SSL model x Downstream architecture")
print("="*60)
print(pivot.to_string())
 
# Also show trainable params
pivot_params = df_results.pivot(index="ssl_model", columns="downstream", values="trainable_params_M")
pivot_params = pivot_params[["linear", "transformer_2L", "transformer_6L"]]
print("\n  Trainable params (M)")
print(pivot_params.to_string())
 
# Ranking per downstream
print("\n  SSL ranking per downstream (by CER, lower = better):")
for ds in ["linear", "transformer_2L", "transformer_6L"]:
    subset = df_results[df_results["downstream"] == ds].sort_values("test_cer")
    ranking = " > ".join(subset["ssl_model"].tolist())
    print(f"  {ds}: {ranking}")
 
print(f"\nAll results saved to {RESULTS_DIR}/")


  CER by SSL model x Downstream architecture
downstream     linear  transformer_2L  transformer_6L
ssl_model                                            
HuBERT-base    0.9023          0.3606          0.3533
mHuBERT-147    0.8220          0.2257          0.2280
wav2vec2-base  0.8318          0.7824          0.3743

  Trainable params (M)
downstream     linear  transformer_2L  transformer_6L
ssl_model                                            
HuBERT-base      0.03            9.09           24.85
mHuBERT-147      0.03            9.09           24.85
wav2vec2-base    0.03            9.09           24.85

  SSL ranking per downstream (by CER, lower = better):
  linear: mHuBERT-147 > wav2vec2-base > HuBERT-base
  transformer_2L: mHuBERT-147 > HuBERT-base > wav2vec2-base
  transformer_6L: mHuBERT-147 > HuBERT-base > wav2vec2-base

All results saved to /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/results/downstream_scaling/
